# Step 14: Descriptive figures (black-and-white, HSR format)

**Inputs:** `data/derived/filtered_data_recovered.csv` (author × paper rows, for the year-level counts) and the outputs of
`step11_cox_dataprep.ipynb` (step 11): `cox_data.csv`, `diag_gender_codes_by_entry_year.csv`.

**Notes**

* black-and-white: grey shades, line styles and hatching instead of colour; width 10.7 cm; PDF + TIFF (300 dpi);
* both year-level panels count **distinct authors** per year (not author × paper rows);
* entry-variable distributions and Kaplan–Meier curves are drawn for the analysis sample of the main specification
  (`ENTRY_MIN`–`ENTRY_MAX`, authors active in ≥ 2 years); set `ENTRY_MIN = 2002` for the 2002–2018 check;
* Kaplan–Meier estimates are computed directly (product-limit estimator), no extra package needed.

| output | content |
|---|---|
| `fig_authors_gender_per_year` | distinct active authors per publication year by gender code (stacked) |
| `fig_female_share_entrants` | female share among new authors per entry year |
| `fig_entry_distributions` | distribution of entry variables by gender (analysis sample) |
| `fig_km_gender` | Kaplan–Meier survival by gender (analysis sample) |
| `fig_km_cohorts` | Kaplan–Meier survival by gender and entry period |
| `data_cox_sample_size.csv` | unique authors (by gender), papers and authorships in the Cox-model sample |

## 0. Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import duckdb

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RAW_CSV = f'{BASE_DIR}/data/derived/filtered_data_recovered.csv'   # output of step 10
DATAPREP_DIR = f'{BASE_DIR}/data/derived'                          # output of step 11
FIG_DIR = f'{BASE_DIR}/figures/descriptives'

ENTRY_MIN, ENTRY_MAX = 1950, 2018     # analysis sample of the main specification
DROP_ONE_YEAR = True                  # main specification: authors active in >= 2 distinct years
GAP = 2                               # dropout definition used for the Kaplan-Meier curves
os.makedirs(FIG_DIR, exist_ok=True)

CM = 1 / 2.54
WIDTH = 10.7 * CM
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'STIX Two Text', 'DejaVu Serif'],
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.0, 'hatch.linewidth': 0.4,
    'savefig.dpi': 300,
})
GENDER_STYLE = {1: dict(color='0.45', linestyle='--', label='female'),
                0: dict(color='black', linestyle='-', label='male')}

def save(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/{name}.tiff', dpi=300, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
    print('saved', name)

## 1. Distinct active authors per publication year, by gender code

Gender codes: F, M (Wiki-Gendersort), UNI (unisex), INI (initials only), UNK (unknown / no name).
Note: `filtered_data_recovered.csv` only contains authors who entered before 2019, so the number of active authors
falls after 2018 by construction (dotted line).

In [ ]:
con = duckdb.connect()
per_year = con.execute(f"""
SELECT
    CAST(PubYear AS INTEGER) AS year,
    CASE WHEN gender IN ('F', 'M', 'UNI', 'INI') THEN gender ELSE 'UNK' END AS gender_code,
    count(DISTINCT AID) AS n_authors
FROM read_csv('{RAW_CSV}', header = true, all_varchar = true)
WHERE AID <> '0'
GROUP BY ALL
ORDER BY ALL
""").df()
per_year.to_csv(f'{FIG_DIR}/data_authors_gender_per_year.csv', index=False)
tab = per_year.pivot(index='year', columns='gender_code', values='n_authors').fillna(0)
tab

In [ ]:
order = ['M', 'F', 'UNI', 'INI', 'UNK']
fills = {'M': dict(color='0.15'), 'F': dict(color='0.55'), 'UNI': dict(color='white', hatch='////', edgecolor='0.3'),
         'INI': dict(color='0.85'), 'UNK': dict(color='white', hatch='....', edgecolor='0.3')}
labels = {'M': 'male', 'F': 'female', 'UNI': 'unisex', 'INI': 'initials only', 'UNK': 'unknown'}

fig, ax = plt.subplots(figsize=(WIDTH, 6.5 * CM))
bottom = np.zeros(len(tab))
for g in order:
    vals = tab[g].values / 1e6
    ax.bar(tab.index, vals, bottom=bottom, width=0.85, linewidth=0.2, label=labels[g], **fills[g])
    bottom += vals
ax.axvline(2018.5, color='black', linestyle=':', linewidth=0.7)
ax.set_ylim(0, bottom.max() * 1.15)
ax.text(2017.5, bottom.max() * 1.13, 'last entry cohort (2018)', ha='right', va='top', fontsize=6)
ax.set_xlabel('publication year')
ax.set_ylabel('active authors (million)')
ax.set_xlim(1949, 2024)
ax.legend(frameon=False, loc='upper left', ncol=1)
fig.tight_layout()
save(fig, 'fig_authors_gender_per_year')

## 2. Female share among new authors per entry year

Share F / (F + M) among authors whose first publication falls in the year. The jump between 2001 and 2002 coincides
with MEDLINE starting to record full first names (see `fig_classification_artifact`); before 2002 the share is
measured on a population selected on career length.

In [ ]:
codes = pd.read_csv(f'{DATAPREP_DIR}/diag_gender_codes_by_entry_year.csv')
ct = codes.pivot(index='begin_year', columns='gender_code', values='n_authors').fillna(0)
fshare = ct['F'] / (ct['F'] + ct['M'])
fshare.rename('female_share_entrants').to_csv(f'{FIG_DIR}/data_female_share_entrants.csv')

fig, ax = plt.subplots(figsize=(WIDTH, 5.5 * CM))
ax.plot(fshare.index, fshare * 100, color='black')
ax.set_xlabel('entry year')
ax.set_ylabel('women among new authors (%)')
ax.grid(axis='y', linewidth=0.3, alpha=0.5)
fig.tight_layout()
save(fig, 'fig_female_share_entrants')
print(fshare.loc[[1950, 1980, 2001, 2002, 2018]].round(3))

## 3. Analysis sample: distribution of entry variables by gender

In [ ]:
cols = ['gender_f', 'begin_year', 'one_year', 'first_year_papers', 'topic_size_entry', 'topic_maturity_entry',
        'female_share_entry', f'event_{GAP}y', f'duration_{GAP}y']
cox = pd.read_csv(f'{DATAPREP_DIR}/cox_data.csv', usecols=cols)
s = cox[(cox.begin_year >= ENTRY_MIN) & (cox.begin_year <= ENTRY_MAX)]
if DROP_ONE_YEAR:
    s = s[s.one_year == 0]
print(f'analysis sample: {len(s):,} authors ({s.gender_f.mean():.1%} women)')

# linear bins for topic size; cap at 99th percentile so the long tail doesn't squash the plot
ts_max = np.ceil(s.topic_size_entry.quantile(0.99) / 100) * 100
panels = [
    ('first_year_papers', 'first-year papers', np.arange(0.5, 11.5, 1), np.arange(1, 11)),
    ('topic_size_entry', 'topic size at entry (papers)', np.linspace(0, ts_max, 41), None),
    ('topic_maturity_entry', 'topic maturity at entry (years)', np.arange(-0.5, 41.5, 1), None),
    ('female_share_entry', 'female share at entry (%)', np.linspace(0, 100, 51), None),
]
fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 8 * CM))
for ax, (var, lab, bins, ticks) in zip(axes.ravel(), panels):
    for g in (0, 1):
        x = s.loc[s.gender_f == g, var] * (100 if var == 'female_share_entry' else 1)
        w = np.ones(len(x)) / len(x) * 100
        st = GENDER_STYLE[g]
        ax.hist(x, bins=bins, weights=w, histtype='step', color=st['color'], linestyle=st['linestyle'],
                linewidth=1.0, label=st['label'])
    if ticks is not None:
        ax.set_xticks(ticks)
    ax.set_xlabel(lab)
    ax.set_ylabel('% of authors')
axes[0, 0].legend(frameon=False)
fig.tight_layout()
save(fig, 'fig_entry_distributions')

summary = s.groupby('gender_f')[[p[0] for p in panels]].agg(['mean', 'median']).round(3)
#summary.to_csv(f'{FIG_DIR}/tab_entry_variables_by_gender.csv')
summary

## 3b. Size of the Cox-model sample: unique authors and papers

Counts for the analysis sample used in the Cox models (same filters as above: `ENTRY_MIN`–`ENTRY_MAX`,
`DROP_ONE_YEAR`). A paper is counted if at least one sample author is among its authors; papers are counted over
the authors' full publication records up to 2023 (not only the entry year). Every author × paper pair is counted once.
For the main specification this gives 5,669,249 authors and 26,779,203 papers (out of 28,611,606 papers in the input data).

In [ ]:
con = duckdb.connect()
cox_sample = con.execute(f"""
WITH sample AS (
    SELECT CAST(AID AS BIGINT) AS aid, gender_f
    FROM read_csv('{DATAPREP_DIR}/cox_data.csv', header = true)
    WHERE begin_year BETWEEN {ENTRY_MIN} AND {ENTRY_MAX}
      {'AND one_year = 0' if DROP_ONE_YEAR else ''}
),
author_paper AS (       -- distinct author x paper pairs in the input data
    SELECT DISTINCT CAST(AID AS BIGINT) AS aid, CAST(PMID AS BIGINT) AS pmid
    FROM read_csv('{RAW_CSV}', header = true, all_varchar = true)
    WHERE AID <> '0'
)
SELECT
    count(DISTINCT s.aid)                                       AS n_authors,
    count(DISTINCT s.aid) FILTER (WHERE s.gender_f = 1)         AS n_women,
    count(DISTINCT s.aid) FILTER (WHERE s.gender_f = 0)         AS n_men,
    count(DISTINCT ap.pmid)                                     AS n_papers,
    count(DISTINCT ap.pmid) FILTER (WHERE s.gender_f = 1)       AS n_papers_with_female_author,
    count(DISTINCT ap.pmid) FILTER (WHERE s.gender_f = 0)       AS n_papers_with_male_author,
    count(*)                                                    AS n_authorships
FROM sample s
JOIN author_paper ap USING (aid)
""").df()
# note: authors without any paper in RAW_CSV would be missing from n_authors (should not occur; compare with len(s) above)
cox_sample.to_csv(f'{FIG_DIR}/data_cox_sample_size.csv', index=False)
cox_sample.T.rename(columns={0: 'count'})

## 4. Kaplan–Meier survival curves

Product-limit estimator for integer durations: S(t) = Π_{u ≤ t} (1 − d_u / n_u),
d_u = dropouts at duration u, n_u = authors still at risk at u (censored authors leave the risk set after their duration).

With the 2-year gap definition and authors active in ≥ 2 distinct years, the earliest possible dropout is at
duration 3 (last paper in year 1 + 2 inactive years), so all curves start at 1 until t = 3 by construction.

In [ ]:
def kaplan_meier(duration, event):
    d = pd.DataFrame({'t': duration, 'e': event})
    deaths = d[d.e == 1].groupby('t').size()
    exits = d.groupby('t').size()
    times = np.sort(d.t.unique())
    at_risk = exits.reindex(times, fill_value=0)[::-1].cumsum()[::-1]
    h = deaths.reindex(times, fill_value=0) / at_risk
    surv = (1 - h).cumprod()
    return pd.DataFrame({'t': times, 'S': surv.values, 'at_risk': at_risk.values})

dur, ev = f'duration_{GAP}y', f'event_{GAP}y'
fig, ax = plt.subplots(figsize=(WIDTH * 0.75, 6 * CM))
km_rows = []
for g in (0, 1):
    sub = s[s.gender_f == g]
    km = kaplan_meier(sub[dur].values, sub[ev].values)
    st = GENDER_STYLE[g]
    ax.step(np.r_[0, km.t], np.r_[1, km.S], where='post', color=st['color'], linestyle=st['linestyle'], label=st['label'])
    km_rows.append(km.assign(gender_f=g))
pd.concat(km_rows).to_csv(f'{FIG_DIR}/data_km_gender.csv', index=False)
ax.set_xlabel('years since first publication')
ax.set_ylabel('share still publishing')
ax.set_ylim(0, 1.02)
ax.grid(linewidth=0.3, alpha=0.5)
ax.legend(frameon=False)
fig.tight_layout()
save(fig, 'fig_km_gender')

In [ ]:
# by entry period (4-year periods within the analysis window)
edges = list(range(ENTRY_MIN, ENTRY_MAX + 1, 4)) if ENTRY_MIN >= 2000 else list(range(ENTRY_MIN, ENTRY_MAX + 1, 10))
periods = [(a, min(b - 1, ENTRY_MAX)) for a, b in zip(edges, edges[1:] + [ENTRY_MAX + 1])]
if periods[-1][1] - periods[-1][0] < 2 and len(periods) > 1:          # merge a short last period
    periods[-2] = (periods[-2][0], periods[-1][1]); periods.pop()
ncol = 2 if len(periods) <= 4 else 3
nrow = int(np.ceil(len(periods) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(WIDTH, 4.2 * CM * nrow), sharey=True, squeeze=False)
for ax, (a, b) in zip(axes.ravel(), periods):
    for g in (0, 1):
        sub = s[(s.gender_f == g) & s.begin_year.between(a, b)]
        km = kaplan_meier(sub[dur].values, sub[ev].values)
        st = GENDER_STYLE[g]
        ax.step(np.r_[0, km.t], np.r_[1, km.S], where='post', color=st['color'], linestyle=st['linestyle'], label=st['label'])
    ax.set_title(f'entry {a}–{b}')
    ax.set_ylim(0, 1.02)
    ax.grid(linewidth=0.3, alpha=0.5)
for ax in axes.ravel()[len(periods):]:
    ax.axis('off')
for ax in axes[:, 0]:
    ax.set_ylabel('share still publishing')
for ax in axes[-1, :]:
    ax.set_xlabel('years since first publication')
axes[0, 0].legend(frameon=False)
fig.tight_layout()
save(fig, 'fig_km_cohorts')